# 04 — Một hàng đầu vào và một mục tiêu

Dự báo log return phiên kế tiếp, sau đó suy ra giá và hướng. Chạy các ô từ trên xuống. Dữ liệu dừng tại 30/09/2025.

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Từ giá thành 32 biến

5 trường giá/volume hiện tại + 23 chỉ báo kỹ thuật + 4 biến nến. Các chỉ báo chỉ dùng phiên hiện tại và quá khứ. `next_log_return` là nhãn, không phải đầu vào.

In [ ]:
from features import load_price_data,build_feature_table,PRICE_ONLY_FEATURE_COLUMNS
raw=load_price_data(ROOT/'data/raw/fpt/fpt_historical_price.csv')
table=build_feature_table(raw)
print('Số đầu vào:',len(PRICE_ONLY_FEATURE_COLUMNS))
display(table.iloc[[-2,-1]].T)

## Một dự báo sinh ra giá và hướng

Nếu log return = ln(1.01), giá 100 thành 101; dấu dương nghĩa là tăng. Giá trị 0 nghĩa là đứng giá, không phải tăng.

In [ ]:
import numpy as np
from metrics import restore_price,direction_from_return
example=np.log([1.01,0.99,1.0])
display(pd.DataFrame({'log_return':example,'predicted_close':restore_price([100,100,100],example),'direction':direction_from_return(example)}))